In [0]:
base = "/Volumes/Adventure_Works/default/raw_data"
target = "Adventure_Works.default"

tables = {
    "fact_sales": ("SalesOrderDetail.csv", [
        "SalesOrderID", "SalesOrderDetailID", "CarrierTrackingNumber", "OrderQty",
        "ProductID", "SpecialOfferID", "UnitPrice", "UnitPriceDiscount",
        "LineTotal", "rowguid", "ModifiedDate"]),
    "dim_product": ("Product.csv", [
        "ProductID", "Name", "ProductNumber", "MakeFlag", "FinishedGoodsFlag",
        "Color", "SafetyStockLevel", "ReorderPoint", "StandardCost", "ListPrice",
        "Size", "SizeUnitMeasureCode", "WeightUnitMeasureCode", "Weight",
        "DaysToManufacture", "ProductLine", "Class", "Style",
        "ProductSubcategoryID", "ProductModelID", "SellStartDate", "SellEndDate",
        "DiscontinuedDate", "rowguid", "ModifiedDate"]),
    "dim_sales_territory": ("SalesTerritory.csv", [
        "TerritoryID", "Name", "CountryRegionCode", "Group", "SalesYTD",
        "SalesLastYear", "CostYTD", "CostLastYear", "rowguid", "ModifiedDate"]),
}

for table, (file, cols) in tables.items():
    df = (spark.read
          .option("header", False)
          .option("sep", "\t")          # files tab-separated hain
          .option("inferSchema", True)  # numbers ko number samjho
          .csv(f"{base}/{file}")
          .toDF(*cols))
    df.write.mode("overwrite").saveAsTable(f"{target}.{table}")
    print(f"{table}: {df.count()} rows saved")

fact_sales: 121317 rows saved
dim_product: 504 rows saved
dim_sales_territory: 10 rows saved


### Exercise 3: Databricks Sidebar
- **Workspace:** Where I store my notebooks, files and folders.
- **Catalog:** Shows all data (catalogs, schemas, tables, volumes) managed by Unity Catalog.
- **Compute:** The machines (clusters / serverless) that run my code.
- **Jobs & Pipelines (Workflows):** Run notebooks automatically on a schedule or as a pipeline.
- **Data Ingestion:** Tools to bring data from outside (files, databases, apps) into Databricks.%md


In [0]:
spark.sql("USE Adventure_Works.default")
display(spark.sql("DESCRIBE fact_sales"))

col_name,data_type,comment
SalesOrderID,int,null
SalesOrderDetailID,int,null
CarrierTrackingNumber,string,null
OrderQty,int,null
ProductID,int,null
SpecialOfferID,int,null
UnitPrice,double,null
UnitPriceDiscount,double,null
LineTotal,double,null
rowguid,string,null


In [0]:
display(spark.sql("DESCRIBE dim_product"))

col_name,data_type,comment
ProductID,int,null
Name,string,null
ProductNumber,string,null
MakeFlag,int,null
FinishedGoodsFlag,int,null
Color,string,null
SafetyStockLevel,int,null
ReorderPoint,int,null
StandardCost,double,null
ListPrice,double,null


In [0]:
display(spark.sql("DESCRIBE dim_sales_territory"))

col_name,data_type,comment
TerritoryID,int,null
Name,string,null
CountryRegionCode,string,null
Group,string,null
SalesYTD,double,null
SalesLastYear,double,null
CostYTD,double,null
CostLastYear,double,null
rowguid,string,null
ModifiedDate,timestamp,null


In [0]:
%sql
SELECT COUNT(*) AS rows FROM Adventure_Works.default.fact_sales

rows
121317


In [0]:
%sql
SELECT COUNT(*) AS rows FROM Adventure_Works.default.dim_product

rows
504


In [0]:
%sql
SELECT COUNT(*) AS rows FROM Adventure_Works.default.dim_product

rows
504


In [0]:
%sql
SELECT COUNT(*) AS rows FROM Adventure_Works.default.dim_sales_territory

rows
10


%md
**Row counts:** fact_sales = 121,317 | dim_product = 504 | dim_sales_territory = 10
The fact table is large (one row per order line), the dimensions are small (one row per product / territory) — this is the expected shape of a star schema.

In [0]:
%sql
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT SalesOrderDetailID) AS distinct_ids
FROM Adventure_Works.default.fact_sales

total_rows,distinct_ids
121317,121317


In [0]:
%sql
SELECT COUNT(*) AS rows_after_left_join
FROM Adventure_Works.default.fact_sales f
LEFT JOIN Adventure_Works.default.dim_product p
  ON f.ProductID = p.ProductID

rows_after_left_join
121317


In [0]:
%sql
SELECT COUNT(*) AS rows_after_right_join
FROM Adventure_Works.default.fact_sales f
RIGHT JOIN Adventure_Works.default.dim_product p
  ON f.ProductID = p.ProductID

rows_after_right_join
121555


In [0]:
%sql
SELECT ProductID, COUNT(*) AS cnt
FROM Adventure_Works.default.dim_product
GROUP BY ProductID
HAVING COUNT(*) > 1

ProductID,cnt


In [0]:
%sql
SELECT TerritoryID, COUNT(*) AS cnt
FROM Adventure_Works.default.dim_sales_territory
GROUP BY TerritoryID
HAVING COUNT(*) > 1

TerritoryID,cnt


%md
### Why a duplicate ProductID double-counts revenue
If `dim_product` had the same ProductID twice, every sales line for that product would match **both** rows in the join, so it would appear twice and `SUM(LineTotal)` would be doubled.
It is **silent**: no error, the query looks correct, and the numbers look plausible. Only validation catches it (row count before vs. after the join, and the duplicate-key check).
**Our result:** LEFT JOIN count = 121,317 (same as fact_sales) and no duplicate keys → no fan-out, the star is safe.

In [0]:
%sql
SELECT * FROM Adventure_Works.default.fact_sales
ORDER BY ModifiedDate DESC
LIMIT 10

SalesOrderID,SalesOrderDetailID,CarrierTrackingNumber,OrderQty,ProductID,SpecialOfferID,UnitPrice,UnitPriceDiscount,LineTotal,rowguid,ModifiedDate
75088,121228,null,1,932,1,24.99,0.0,24.99,72C8BC51-5066-49A8-A20D-3BF7297C5B59,2025-06-29T00:00:00.000Z
75088,121227,null,1,922,1,3.99,0.0,3.99,FCFA5A5C-D6E4-42B8-BE60-31873C3A2077,2025-06-29T00:00:00.000Z
75085,121224,null,1,877,1,7.95,0.0,7.95,35E2D64E-BC36-42E2-9C8E-5BAE7524D84F,2025-06-29T00:00:00.000Z
75088,121230,null,1,713,1,49.99,0.0,49.99,5C085429-02B3-45C9-ADC8-ED18C64C3589,2025-06-29T00:00:00.000Z
75087,121226,null,1,707,1,34.99,0.0,34.99,D27BF866-D64B-458B-AFF3-51C542F14B73,2025-06-29T00:00:00.000Z
75086,121225,null,1,877,1,7.95,0.0,7.95,8B7B0121-CEF8-41B7-B58C-F465F972EDFD,2025-06-29T00:00:00.000Z
75084,121222,null,1,876,1,120.0,0.0,120.0,903DB72A-1A16-4220-9500-A1F6235FC355,2025-06-29T00:00:00.000Z
75085,121223,null,1,712,1,8.99,0.0,8.99,D9F59E2A-FCDA-4A02-AD25-016BE258CFBE,2025-06-29T00:00:00.000Z
75088,121229,null,1,711,1,34.99,0.0,34.99,8EDB26BE-B63A-42DD-BC08-73972445D4D0,2025-06-29T00:00:00.000Z
75089,121231,null,1,931,1,21.49,0.0,21.49,EA730EFD-9589-4DB4-9909-7C954D1F88F9,2025-06-29T00:00:00.000Z


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM Adventure_Works.default.fact_sales) AS total_rows,
  (SELECT COUNT(DISTINCT ProductID) FROM Adventure_Works.default.fact_sales) AS products_sold,
  (SELECT COUNT(DISTINCT ProductID) FROM Adventure_Works.default.dim_product) AS products_total

total_rows,products_sold,products_total
121317,266,504


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM Adventure_Works.default.fact_sales) AS total_rows,
  (SELECT COUNT(DISTINCT ProductID) FROM Adventure_Works.default.fact_sales) AS products_sold,
  (SELECT COUNT(DISTINCT ProductID) FROM Adventure_Works.default.dim_product) AS products_total

total_rows,products_sold,products_total
121317,266,504


In [0]:
%sql
SELECT p.Name, ROUND(SUM(f.LineTotal), 2) AS revenue
FROM Adventure_Works.default.fact_sales f
JOIN Adventure_Works.default.dim_product p ON f.ProductID = p.ProductID
GROUP BY p.Name
ORDER BY revenue DESC
LIMIT 10

Name,revenue
"Mountain-200 Black, 38",4400592.8
"Mountain-200 Black, 42",4009494.76
"Mountain-200 Silver, 38",3693678.03
"Mountain-200 Silver, 42",3438478.86
"Mountain-200 Silver, 46",3434256.94
"Mountain-200 Black, 46",3309673.22
"Road-250 Black, 44",2516857.31
"Road-250 Black, 48",2347655.95
"Road-250 Black, 52",2012447.78
"Road-150 Red, 56",1847818.63


In [0]:
%sql
SELECT COUNT(*) AS bad_rows
FROM Adventure_Works.default.fact_sales
WHERE LineTotal IS NULL OR LineTotal < 0

bad_rows
0


In [0]:
%sql
SELECT p.Name, ROUND(SUM(f.LineTotal), 2) AS revenue
FROM Adventure_Works.default.fact_sales f
JOIN Adventure_Works.default.dim_product p ON f.ProductID = p.ProductID
GROUP BY p.Name
ORDER BY revenue DESC
LIMIT 5

Name,revenue
"Mountain-200 Black, 38",4400592.8
"Mountain-200 Black, 42",4009494.76
"Mountain-200 Silver, 38",3693678.03
"Mountain-200 Silver, 42",3438478.86
"Mountain-200 Silver, 46",3434256.94


Databricks visualization. Run in Databricks to view.

%md
**Insight:** The top 5 revenue products are all **Mountain-200** bikes (each ~3.4–4.4M).
 Revenue is driven by high-price bikes, not by accessories.


In [0]:
%sql
SELECT 'fact_sales' AS tbl, COUNT(*) AS rows, COUNT(DISTINCT SalesOrderDetailID) AS distinct_keys FROM Adventure_Works.default.fact_sales
UNION ALL
SELECT 'dim_product', COUNT(*), COUNT(DISTINCT ProductID) FROM Adventure_Works.default.dim_product
UNION ALL
SELECT 'dim_sales_territory', COUNT(*), COUNT(DISTINCT TerritoryID) FROM Adventure_Works.default.dim_sales_territory

tbl,rows,distinct_keys
fact_sales,121317,121317
dim_product,504,504
dim_sales_territory,10,10


In [0]:
%sql
SELECT COUNT(*) AS orphan_sales
FROM Adventure_Works.default.fact_sales f
LEFT JOIN Adventure_Works.default.dim_product p ON f.ProductID = p.ProductID
WHERE p.ProductID IS NULL

orphan_sales
0


In [0]:
%sql
SELECT COUNT(*) AS never_sold
FROM Adventure_Works.default.dim_product p
LEFT JOIN Adventure_Works.default.fact_sales f ON p.ProductID = f.ProductID
WHERE f.ProductID IS NULL

never_sold
238


**Orphans = 0:** every sale points to an existing product → referential integrity is OK.
**Never sold = 238:** these products exist in the catalog but have no sales (e.g. components or discontinued items). This is not an error — it's why the RIGHT JOIN in Exercise 4 returned 238 extra rows.
The difference: an orphan is a *fact without a dimension* (a real data problem), an unsold product is a *dimension without facts* (normal).

In [0]:
%sql
SELECT p.Name, COUNT(*) AS order_lines, SUM(f.OrderQty) AS units
FROM Adventure_Works.default.fact_sales f
JOIN Adventure_Works.default.dim_product p ON f.ProductID = p.ProductID
GROUP BY p.Name
ORDER BY order_lines DESC
LIMIT 10

Name,order_lines,units
Water Bottle - 30 oz.,4688,6815
AWC Logo Cap,3382,8311
Patch Kit/8 Patches,3354,3865
Mountain Tire Tube,3095,3095
"Sport-100 Helmet, Blue",3090,6743
"Sport-100 Helmet, Red",3083,6266
"Sport-100 Helmet, Black",3007,6532
Road Tire Tube,2376,2376
Fender Set - Mountain,2121,2121
Mountain Bottle Cage,2025,2025


In [0]:
%sql
SELECT p.ProductLine, ROUND(SUM(f.LineTotal), 0) AS revenue
FROM Adventure_Works.default.fact_sales f
JOIN Adventure_Works.default.dim_product p ON f.ProductID = p.ProductID
GROUP BY p.ProductLine
ORDER BY revenue DESC

ProductLine,revenue
R,4.8295262E7
M,4.2458404E7
T,1.6010837E7
S,2541630.0
null,540249.0


%md
**Insight:** Accessories (water bottles, caps) are ordered **most often**, but bikes (Road & Mountain lines) bring **most of the revenue**. Frequency ≠ revenue.
